[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/2-4_%ED%8C%90%EC%A0%95%EA%B8%B0%EC%A4%80%EC%9D%84_%EB%B9%84%EC%9A%A9%EC%9C%BC%EB%A1%9C_%EC%A0%95%ED%95%98%EA%B8%B0.ipynb)

# 2-4 · 판정 기준을 비용으로 정하기 — 심한 누설을 놓치지 않으려면

**⏱ 90분** · 모듈 2 마무리: 센서 → 특징 → 이상탐지 → 고장진단 → **판정 기준**

## 🎯 이번 시간의 질문

> 정확도 97%의 진단 모델이 **심한 누설 123개 중 8개**를 "약한 누설"로 판정했습니다. 판정 기준을 바꾸면 이 8개를 잡을 수 있을까요? **얼마나** 바꿔야 할까요?

2-3의 랜덤 포레스트는 가장 확률이 높은 상태를 답으로 골랐습니다. 하지만 심한 누설을 놓치면 **펌프가 파손되고 라인이 멈춥니다**. 반대로 "심함"이라고 너무 쉽게 판정하면 멀쩡한 펌프를 세우고 점검하게 됩니다.
1-3에서 SMART 경보 기준을 정했던 방법 그대로, **두 실수에 비용을 붙여 가장 싼 기준**을 찾습니다. 그리고 현장에서 센서가 조금 틀어졌을 때 그 기준이 견디는지도 확인합니다.

**이번 시간에 익힐 패턴:** `predict_proba`로 확률 보기 · 1-3과 같은 비용 계산 · P4 선그래프로 기준 비교

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/hydraulic_labels.csv",
             "data/hydraulic_features.csv",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

## 1. 준비 (▶) — 2-3의 랜덤 포레스트

2-3과 같은 데이터·특징으로 펌프 누설 진단 모델을 학습해 둡니다.

In [ ]:
# ▶ 그대로 실행하세요
from sklearn.ensemble import RandomForestClassifier

labels = pd.read_csv('data/hydraulic_labels.csv')
features = pd.read_csv('data/hydraulic_features.csv')
data = labels.merge(features, on='cycle_id')
train = data[data['split'] == 'train']
test = data[data['split'] == 'test']
mean_cols = ['PS1_mean', 'PS2_mean', 'PS3_mean', 'PS4_mean', 'PS5_mean', 'PS6_mean', 'EPS1_mean',
             'FS1_mean', 'FS2_mean', 'TS1_mean', 'TS2_mean', 'TS3_mean', 'TS4_mean', 'VS1_mean']
X_test = test[mean_cols]
y_test = test['pump_leakage']

forest = RandomForestClassifier(random_state=0)
forest.fit(train[mean_cols], train['pump_leakage'])
print('정확도:', round((forest.predict(X_test) == y_test).mean() * 100, 1), '%')

## 2. 모델의 답을 확률로 보기

`predict()`는 답 하나만 주지만, `predict_proba()`는 **상태별 확률**을 줍니다. 랜덤 포레스트에서 확률은 **나무 100그루 중 그 상태에 투표한 비율**입니다.
결과는 사이클마다 [없음, 약함, 심함] 세 칸이므로, 세 번째 칸(`[:, 2]`)이 "심한 누설일 확률"입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
proba = forest.predict_proba(X_test)
p_severe = proba[:, 2]
print(proba[:3].round(2))
print(p_severe[:3].round(2))
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 첫 사이클은 [0, 0.46, 0.54]: 나무 100그루 중 54그루가 "심함", 46그루가 "약함"에 투표했습니다. 두 번째 사이클은 92그루가 "심함"입니다.
`predict()`는 **가장 큰 확률**을 답으로 고르므로, 사실상 "심함 확률이 절반쯤을 넘어야 심함"이라는 기준을 쓰고 있는 셈입니다. 첫 사이클처럼 **의견이 갈리는 사이클**은 기준을 조금만 바꿔도 판정이 뒤집힙니다.

실제 상태별로 "심함 확률"의 분포를 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
for state in [0, 1, 2]:
    plt.hist(p_severe[y_test == state], bins=20, alpha=0.6, label=f'실제 누설 {state}')
plt.axvline(0.5, color='red', linestyle='--', label='기준 0.5')
plt.xlabel('모델이 본 "심한 누설" 확률')
plt.ylabel('사이클 수')
plt.legend()
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 실제 심한 누설(초록)은 대부분 0.7 이상이지만, **몇 개는 0.2~0.5에** 있습니다. 이것이 2-3에서 놓친 8개입니다. 실제 누설 없음(파랑)·약함(주황)은 대부분 0.2 아래에 있습니다.
기준(빨간 선)을 **왼쪽으로 옮기면** 놓친 심한 누설을 잡을 수 있고, 너무 옮기면 약한 누설도 "심함"이 됩니다.

## 3. 기준 0.5 vs 0.2 — 네 칸으로 비교

"심함 확률 ≥ 기준"이면 **심한 누설 경보**(라인 정지 · 펌프 점검)를 냅니다. 1-2·2-2와 같은 네 칸으로 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
is_severe = y_test == 2
alarm = p_severe >= 0.5
pd.crosstab(is_severe, alarm, rownames=['실제 심한 누설'], colnames=['경보'])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 기준 0.5: 심한 누설 123개 중 115개 경보, **8개 놓침**, 헛경보 0개(2-3의 결과와 같음).

**같은 코드에서 0.5를 0.2로만** 바꿔 실행해 보세요.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
alarm = p_severe >= 0.2
pd.crosstab(is_severe, alarm, rownames=['실제 심한 누설'], colnames=['경보'])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 기준 0.2: **놓침 0개**, 대신 헛경보 4개(심하지 않은 누설을 심함으로 판정). 어느 쪽이 나을까요? 두 실수의 **값어치**가 다르므로 비용으로 비교합니다.

## 4. 비용 모델 — 1-3과 같은 구조

| 기호 | 뜻 | 이번 가정 |
|---|---|---|
| `Cp` | **심한 누설 경보 1건의 대응 비용** (라인 정지 · 펌프 점검) | 1 (기준 단위) |
| `Cf` | **놓친 심한 누설 1건의 비용** (펌프 파손 · 긴급 수리 · 긴 정지) | 10 (점검의 10배로 가정) |

**총비용 = Cp × 경보 수 + Cf × 놓친 심한 누설 수** (1-3의 `Cp × 경보 대수 + Cf × 막지 못한 고장`과 같은 식)

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
Cp = 1
Cf = 10
for th in [0.5, 0.2]:
    alarm = p_severe >= th
    missed = (is_severe & ~alarm).sum()
    print('기준', th, '→ 경보', alarm.sum(), '· 놓침', missed, '· 총비용', Cp * alarm.sum() + Cf * missed)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 기준 0.5: 경보 115, 놓침 8, 총비용 **195** / 기준 0.2: 경보 127, 놓침 0, 총비용 **127**.

💬 경보(점검)는 12건 늘었지만 놓친 심한 누설 8건(80)을 없애서 총비용이 **35% 줄었습니다.** 그럼 0.2가 최선일까요? 기준을 여러 개 비교합니다.

### ▶ 기준별 비용표 함수

1-3의 `cost_table`과 같은 역할입니다. 기준 0.1 ~ 0.9마다 경보 수, 놓침, 헛경보, 총비용을 계산합니다.

In [ ]:
# ▶ 그대로 실행하세요
def alarm_cost(prob, is_bad, Cp=1, Cf=10):
    """기준(0.1~0.9)별 경보 수·놓침·헛경보·총비용 표."""
    prob, is_bad = np.asarray(prob), np.asarray(is_bad)
    rows = []
    for th in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        alarm = prob >= th
        missed = (is_bad & ~alarm).sum()
        rows.append({'기준': th, '경보': alarm.sum(), '놓침': missed,
                     '헛경보': (alarm & ~is_bad).sum(), '총비용': Cp * alarm.sum() + Cf * missed})
    return pd.DataFrame(rows)

costs = alarm_cost(p_severe, is_severe, Cp=1, Cf=10)
costs

**P4 · 선그래프** — 기준에 따른 총비용을 그립니다(1-3의 비용 곡선과 같은 그림).

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
plt.plot(costs['기준'], costs['총비용'], marker='o')
plt.xlabel('"심한 누설" 판정 기준 (확률)')
plt.ylabel('총비용 (점검 1건 = 1)')
plt.title('판정 기준별 총비용 (Cf = 10)')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 1-3처럼 **U자에 가까운 모양**입니다.
- 기준 0.1: 헛경보 19건이 쌓여 비쌉니다(142).
- 기준 0.2에서 **최소 127**: 놓침 0, 헛경보 4.
- 기준 0.5 이상: 놓침이 늘어 비용이 빠르게 오릅니다(0.9에서는 41개 놓침, 492).

모델을 바꾸지 않고 **판정 기준만** 바꿔서 심한 누설을 하나도 놓치지 않게 됐습니다. 모델의 "가장 큰 확률"을 그대로 답으로 쓰는 것이 운영에 맞는 선택은 아닐 수 있습니다. **기준은 모델이 아니라 비용이 정합니다**(1-3과 같은 결론).

---
## 5. 실습 과제 (45분)

### 📝 과제 1 · 놓침 비용이 달라지면?
`Cf`를 **1, 2, 5, 20**으로 바꿔 최적 기준과 총비용을 찾으세요. 실행 전에 Cf가 커지면 기준이 어느 쪽으로 움직일지 예상해 보세요.

| Cf | 최적 기준 | 경보 | 놓침 | 총비용 | 기준 0.5일 때 총비용 |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 5 | | | | | |
| 10 | 0.2 | 127 | 0 | 127 | 195 |
| 20 | | | | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: c = alarm_cost(p_severe, is_severe, Cp=1, Cf=5)  →  c.sort_values('총비용').head(3)


✏️ **나의 답:**

### 📝 과제 2 · 축압기 "고장 직전" 경보에 적용하기
같은 방법을 **축압기 고장 직전(90 bar)** 경보에 적용하세요. 축압기 진단은 2-3에서 87.5%로 펌프보다 어려웠습니다.

1. 축압기 모델 학습 → `forest_acc = RandomForestClassifier(random_state=0)`, `forest_acc.fit(train[mean_cols], train['accumulator'])`
2. "90 bar일 확률" → 상태값이 작은 순서로 열이 놓이므로 **첫 번째 칸**: `p90 = forest_acc.predict_proba(X_test)[:, 0]`
3. `alarm_cost(p90, test['accumulator'] == 90, Cp=1, Cf=10)`

| | 기준 0.5 | 최적 기준 |
|---|---|---|
| 기준값 | 0.5 | |
| 경보 · 놓침 · 헛경보 | | |
| 총비용 | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 위 1~3단계를 차례로 입력합니다.


✏️ **나의 답:** 펌프와 비교해 무엇이 다른가요?

### 📝 과제 3 · 유량계가 조금 틀어진다면? (운영 위험)
현장 센서는 시간이 지나면 영점이 조금씩 어긋납니다. 유량계 `FS1`이 실제보다 **0.05, 0.1, 0.2 L/min** 크게 읽힌다고 가정하고, 기준 **0.5와 0.2**에서 심한 누설 놓침이 어떻게 변하는지 표를 채우세요.

⚠️ 원래 `X_test`를 바꾸지 않도록 `.copy()`로 복사본을 만들어 값을 더합니다.

| 유량계 오차 (L/min) | 놓침 (기준 0.5) | 놓침 (기준 0.2) |
|---|---|---|
| 0 | 8 | 0 |
| 0.05 | | |
| 0.1 | | |
| 0.2 | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: X_drift = X_test.copy();  X_drift['FS1_mean'] = X_drift['FS1_mean'] + 0.1
# p = forest.predict_proba(X_drift)[:, 2]  →  alarm_cost(p, is_severe) 의 기준 0.2, 0.5 줄을 봅니다.


✏️ **나의 답:** 어느 기준이 센서 오차에 더 잘 견디나요? 왜 그럴까요? (2-3의 결정나무 규칙 "유량 ≤ 6.44 → 심함"을 떠올려 보세요)

### 📝 과제 4 · 정비팀장에게 보내는 3줄 보고
모듈 2의 결과로 **이 유압 장치의 감시·진단 방안**을 3~4문장으로 제안하세요. ① 이상탐지와 진단을 각각 어디에 쓸지 ② 심한 누설 판정 기준과 근거 숫자 ③ 가정과 운영 위험을 포함합니다.

✏️ **나의 답:**

---
## ✅ 모듈 2 정리

| 단계 | 한 일 | 핵심 |
|---|---|---|
| 2-1 특징 | 원신호 → 특징, **구분력**으로 비교 | 냉각기 13 · 펌프·밸브·축압기 약 1. 좋은 특징을 만들면 밸브 1.2 → 8.9 |
| 2-2 이상탐지 | 정상 데이터만으로 z-점수·Isolation Forest | 구분력 높은 냉각기는 100%, 펌프 누설은 20% (기준을 낮추면 86% · 오경보 29%) |
| 2-3 고장진단 | 정답으로 결정나무·랜덤 포레스트 학습 | 펌프 97%. 정확도가 높아도 가장 위험한 고장을 놓칠 수 있다 |
| 2-4 판정 기준 | 확률 기준을 비용으로 선택, 센서 오차 점검 | 기준 0.5 → 0.2: 놓침 8 → 0, 총비용 195 → 127. **기준은 비용이 정한다** |

모듈 1과 모듈 2에서 반복된 원칙: **네 칸(탐지·놓침·헛경보·정상)으로 평가 → 교환 관계 확인 → 비용으로 기준 선택 → 가정과 위험을 적기.**

지금까지는 "**지금** 어떤 상태인가"를 판단했습니다. 정비를 계획하려면 "**앞으로 얼마나** 버틸까"도 알아야 합니다. → 모듈 3 · **잔여수명(RUL)**